# 10 — Derive artifacts

Generate documentation, diagrams, job templates and service/package metadata from one authoritative contract.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/10-derive-artifacts/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/10-derive-artifacts/lesson.md) · [Exercise](../modules/10-derive-artifacts/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/10-derive-artifacts"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Derive genuine projections

Run all course generators into this lab's scratch directory. Markdown/Jinja documentation templates are generated through the repository compatibility adapter; PlantUML source generation does not need a rendering service. These outputs are projections of the contract, not handwritten replacements.


In [ ]:
(
set -euo pipefail
cd "$WORKSPACE"
bash scripts/derive.sh reference/waterbodies.cwl "$LAB_DIR/derived" > "$LAB_DIR/derive.log" 2>&1
cat "$LAB_DIR/derive.log"
find "$LAB_DIR/derived" -type f
)


## Compare an input template with runnable inputs

A generated template describes ports but still needs real data locations. Explain why a URI to an Item JSON cannot replace the staged `Directory` required by this workflow.


In [ ]:
(
set -euo pipefail
cat derived/inputs.yaml
cat "$WORKSPACE/reference/inputs.yaml"
python - <<'PYTHON'
from pathlib import Path
import json
for filename in ['ogc/processes.json', 'oci/annotations.json', 'publication/codemeta.json']:
    print(filename)
    print(json.dumps(json.loads((Path('derived') / filename).read_text()), indent=2))
for pattern in ['docs/**/*.md.jinja', 'diagrams/**/*.puml']:
    paths = list(Path('derived').glob(pattern))
    assert paths, f'Missing projection: {pattern}'
    print(paths[0], paths[0].read_text()[:1800])
PYTHON
)


## Trace one port across projections

Locate `aoi` in the CWL, generated input template and OGC description. Describe which artifact expresses validation, which supplies a runtime value, and which documents the interface. Editing a projection does not update its source contract.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
for path in [Path('derived/inputs.yaml'), Path('derived/ogc/processes.json')]:
    content = path.read_text()
    assert 'aoi' in content
    print(path, 'contains the AOI port')
assert '$manifest' in Path('derived/oci/annotations.json').read_text()
print('All projections are available for comparison')
PYTHON
)


## Record your findings

**Expected outcome:** The real generators emit Markdown/Jinja templates, PlantUML, input YAML, OGC JSON, CodeMeta and OCI metadata without editing committed reference outputs.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/10-derive-artifacts/solution.md) after the activity. Return to [the lab index](README.md).
